In [1]:
from TinyShakespeareDataModule import TinyShakespeareDataModule
import torch
import lightning as L
from pytorch_lightning.loggers import TensorBoardLogger
from MyOwnTransformer import MyTransformer

In [2]:
seq_length = 64
heads = 4
d_model = 128
batch_size = 64
layers = 4

# create lightning data module instance
tiny_shakespeare_datamodule = TinyShakespeareDataModule(seq_length, batch_size)
vocab_size = tiny_shakespeare_datamodule.vocab_size

# create my transformer instance
tiny_shakespeare_transformer = MyTransformer(vocab_size, d_model, seq_length * 2, seq_length, layers, heads)

In [3]:
checkpoint_callback = L.pytorch.callbacks.ModelCheckpoint(
        dirpath = "checkpoints",
        filename = "tinyshakespeare-{epoch:02d}-{val_loss:.4f}",
        save_top_k = 3,  # keep best 3 models
        mode = "min",
        save_last = True,
        monitor = "val_loss"
    )
transformer_trainer = L.Trainer(
        accelerator = "auto",
        devices = 1,
        enable_progress_bar = True,
        callbacks=[checkpoint_callback],
        gradient_clip_val = 1.0)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [4]:
loss_results = transformer_trainer.test(tiny_shakespeare_transformer, tiny_shakespeare_datamodule, ckpt_path = "last" )

Restoring states from the checkpoint path at /Users/botang/Documents/Python_WS/My1stMLPrj/TransformerFromScratch/checkpoints/last.ckpt
/Users/botang/Documents/Conda3/envs/MyCondaEnv/lib/python3.11/site-packages/lightning/pytorch/trainer/call.py:282: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val_loss', 'mode': 'min'}"].
Loaded model weights from the checkpoint at /Users/botang/Documents/Python_WS/My1stMLPrj/TransformerFromScratch/checkpoints/last.ckpt
/Users/botang/Documents/Conda3/envs/MyCondaEnv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 14/14 [00:00<00:00, 14.39it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    1.6087130308151245     │
└───────────────────────────┴───────────────────────────┘

In [5]:
import math
loss_v = math.exp(loss_results[0]["test_loss"])
print(f"On average, the model is as uncertain as choosing between {loss_v:.2f} characters.")

On average, the model is as uncertain as choosing between 5.00 characters.


In [7]:
prompt = 'I will eat apple today' 

chtoidx = tiny_shakespeare_datamodule.char_to_idx
idxtoch = tiny_shakespeare_datamodule.idx_to_char

newgen = tiny_shakespeare_transformer.generate(prompt, chtoidx, idxtoch, top_k = 5)
print(newgen)

I will eat apple today terror,
That they were now by any most strange.

LUCIO:
Then, till thought this time is strength, take my lady;
And be provost thy lady wilt, they his cheek.

GLOUCESTER:
Therefore the man thousand spirits of this people;
To her be not hunts to be troubled again.

QUEEN MARGARET:
It is a concluded, be pretty such a too more;
But in my prisoner tears and the state again
A curaity of thee: i' this the soul discorned;
The better's life than whereof that you have
A barded o' the whipped side with his sovereign.

QUEEN ELIZABETH:
The letters than his houses the heart:
But they have durst not their suns and his brothers,
She hath deserved his lord found her bash'd,
And these happy desires sorrows well.

DUKE OF YORK:
I have night so.

QUEEN MARGARET:
In that, so I may more, and this it stands to be
Than he is to the people, where is the children?

LORD WILLOUGHBY:
My lord, I do say to see thee, that with her brings
With thy brother's soul should be made it banish'd.

D

In [9]:
prompt = "It is a concluded, be pretty such a too more; But in my prisoner tears and the state again A curaity of thee: i' this the soul discorned; The better's life than whereof that you have A barded o' the whipped side with his sovereign."
chtoidx = tiny_shakespeare_datamodule.char_to_idx
idxtoch = tiny_shakespeare_datamodule.idx_to_char

newgen = tiny_shakespeare_transformer.generate(prompt, chtoidx, idxtoch, top_k = 5)
print(newgen)


It is a concluded, be pretty such a too more; But in my prisoner tears and the state again A curaity of thee: i' this the soul discorned; The better's life than whereof that you have A barded o' the whipped side with his sovereign.
And then the liest that wherefore he is here,
When I may be no so deverer'd it thence;
And so I all thee, for I have death'd in the seas.
Thou hast not but thy head and hanging how,
Are by the best days in the creature of the coron:
What's there!

MENENIUS:
Will then, I say, if thou be created, and were so
They will not be past of thy consul: it is thy house.

LUCIO:
True can all to the succe that he did break his church,
And sent his childish is dead; and that would hear him where,
And safe this trumpets and seems and tears:
And he have noble in the chair of his choice,
And that he was not to thy father.

KING EDWARD IV:
Not so did the less of them, and tell me for me.

GLOUCESTER:
It will thy leave.

KING RICHARD II:
Ay therefore, my lord. I have so tribun